# Why 1 % hit inefficiency hurts the line comb — and the fix

The 2×2 showed the QSVT comb's one cost: at 1 % hit drop the efficiency drifts to
~93 % at T = 400 (false rate still 0). This notebook shows **why**, segment by
segment, and then **fixes most of it**:

1. a dropped hit breaks a 5-hit track into **fragments** — $P_3$, $P_2$ or $P_1$
   chains — whose eigenvalues sit exactly **off** the four $P_4$ comb lines, so the
   comb (correctly, from its point of view) erases them;
2. re-admitting the fragment lines spectrally also re-admits their **degenerate
   false twins** (longer false chains share those eigenvalues) — a polynomial
   cannot tell them apart;
3. but a **classical occupancy gate** can: a *true fragment's* hits belong to a
   broken track and are used by **no other active segment**, while a *false
   chain's* hits are shared with the active parent tracks. Gating the recovered
   set on hit-sharing restores the efficiency at ~zero false cost.

Event under the microscope: γ = 3, T = 400, rep 0, hit drop 1 % (the stored
campaign config). τ = 0.35.

In [1]:
import sys, collections
for p in ("/data/bfys/gscriven/Quantum_Track_Reconstruction/Toy_Characterisation/_shared",
          "/data/bfys/gscriven/LHCb_VeLo_Toy_Model/src"):
    if p not in sys.path: sys.path.insert(0,p)
from pathlib import Path
import numpy as np, pandas as pd, scipy.sparse as sp
from scipy.sparse.csgraph import connected_components
import matplotlib.pyplot as plt
import qtrk_pipeline as qp
from lhcb_velo_toy.solvers.quantum import QSVT, design_line_comb_inverse
plt.rcParams.update({"figure.dpi":110,"font.size":11,"axes.grid":True,"grid.alpha":0.3})
OUT=Path("/data/bfys/gscriven/Quantum_Track_Reconstruction/QSVT/Segment_level_studies/outputs/hit_drop"); OUT.mkdir(parents=True,exist_ok=True)
G,D=3.0,1.0; S=G+D; TAU=qp.threshold_for(G,D)
P4L=np.array([S-2*np.cos(k*np.pi/5) for k in (1,2,3,4)])      # 2.382 3.382 4.618 5.618
P3L=np.array([S-np.sqrt(2),S,S+np.sqrt(2)])                   # 2.586 4.0   5.414
P2L=np.array([S-1.0,S+1.0]); P1L=np.array([S])

def load(T,drop,rep=0):
    ev=qp.ensure_event(n_trk=T,rep=rep,sigma_scatt=1e-4,sigma_res=0.0,phi_max=0.2,hit_ineff=drop)[0]
    ham=qp.build_hamiltonian(ev,epsilon=0.002,gamma=G,delta=D)
    truth=np.asarray(qp.truth_from_event(ev),bool)
    A=ham.A.tocsr(); n=ham.n_segments
    Cm=(S*sp.identity(n,format='csr')-A); Cm.setdiag(0); Cm.eliminate_zeros()
    Cm=(abs(Cm)>1e-9).astype(np.int8)
    _,lab=connected_components(Cm,directed=False); deg=np.asarray(Cm.sum(1)).ravel()
    return ev,ham,A,n,truth,lab,deg

def true_classes(truth,lab,deg):
    """cluster class of every TRUE segment: pure-true chain P_k, mixed, or tangle."""
    cls=np.empty(truth.sum(),object)
    tidx=np.where(truth)[0]
    cache={}
    for j,i in enumerate(tidx):
        c=lab[i]
        if c not in cache:
            mem=np.where(lab==c)[0]; k=len(mem)
            if deg[mem].max(initial=0)>2: cache[c]="tangle"
            elif truth[mem].all(): cache[c]=f"P{min(k,5)}-true" if k<=4 else "P5+-true"
            else: cache[c]="mixed chain"
        cls[j]=cache[c]
    return cls,tidx

## 1. What the drop does to the true segments — the fragment census

Each track is 5 hits → 4 segments (a $P_4$ chain). Dropping one hit produces, with
equal probability per hit:
- an **end** hit (2/5): a 4-hit track → **$P_3$ fragment** (3 true segments),
- hit 2 or 4 (2/5): an orphan hit + a 3-hit run → **$P_2$ fragment** (2 segments),
- the **middle** hit (1/5): two 2-hit stubs → **2 × $P_1$** (2 isolated true segments).

At 1 % drop, $1-0.99^5 \approx 4.9\,\%$ of tracks are affected.

In [2]:
census={}
for tag,drop in (("clean",0.0),("1% drop",0.01)):
    ev,ham,A,n,truth,lab,deg=load(400,drop)
    cls,tidx=true_classes(truth,lab,deg)
    census[tag]=collections.Counter(cls)
    if drop>0: EV,HAM,Aq,N,TRUTH,LAB,DEG=ev,ham,A,n,truth,lab,deg; CLS,TIDX=cls,tidx
CT=pd.DataFrame(census).fillna(0).astype(int)
order=["P4-true","P3-true","P2-true","P1-true","mixed chain","tangle"]
CT=CT.reindex([o for o in order if o in CT.index])
print(CT.to_string()); print()
print("affected tracks expectation at 1%:", f"{(1-0.99**5)*100:.1f}% of 400 = {(1-0.99**5)*400:.0f} tracks")

         clean  1% drop
P4-true   1564     1448
P3-true      0       27
P2-true      0       22
P1-true      0       10
tangle      36       51

affected tracks expectation at 1%: 4.9% of 400 = 20 tracks


## 2. The schematic — what a dropped hit does to a track

In [3]:
fig,axs=plt.subplots(4,1,figsize=(11,7.2),sharex=True)
LAY=np.arange(1,6)
def track(ax,hits,segs,col,title,lam):
    ax.scatter(hits,[0]*len(hits),s=130,color="k",zorder=3)
    for (a,b) in segs: ax.plot([a,b],[0,0],color=col,lw=5,alpha=0.85,zorder=2)
    for L in LAY: ax.axvline(L,color="#bbb",lw=0.8,zorder=1)
    ax.set_yticks([]); ax.set_ylim(-1,1); ax.set_xlim(0.6,5.4)
    ax.text(0.62,0.55,title,fontsize=10.5,fontweight="bold")
    ax.text(5.38,0.55,lam,fontsize=9.5,ha="right",family="monospace")
    ax.grid(False)
track(axs[0],[1,2,3,4,5],[(1,2),(2,3),(3,4),(4,5)],"#1b7837",
      "intact track  →  P4 chain (4 true segments)","λ = {2.382, 3.382, 4.618, 5.618}  ON the comb ✓")
track(axs[1],[1,2,3,4],[(1,2),(2,3),(3,4)],"#e08214",
      "END hit dropped (2/5)  →  P3 fragment","λ = {2.586, 4.0, 5.414}  off the comb → erased")
track(axs[2],[1,3,4,5],[(3,4),(4,5)],"#d6604d",
      "hit 2 dropped (2/5)  →  orphan + P2 fragment","λ = {3.0, 5.0}  off the comb → erased")
track(axs[3],[1,2,4,5],[(1,2),(4,5)],"#b2182b",
      "MIDDLE hit dropped (1/5)  →  2 × P1 stubs","λ = 4.0  = the isolated-FALSE line → unrecoverable")
axs[3].set_xlabel("detector layer")
axs[3].set_xticks(LAY)
fig.suptitle("One dropped hit per track: the surviving true segments form shorter chains\nwhose eigenvalues are no longer on the four P4 comb lines",
             fontsize=12,fontweight="bold")
fig.tight_layout(rect=[0,0,1,0.93])
for ext,dpi in (("pdf",600),("png",300)): fig.savefig(OUT/f"drop_schematic.{ext}",dpi=dpi,bbox_inches="tight",facecolor="white")
plt.show(); print("saved drop_schematic")

saved drop_schematic


## 3. The spectral picture and the measured per-class efficiency

The comb passes the four $P_4$ lines; the fragment eigenvalues fall in its
stop-band — including $P_1$ at λ = 4.0, **exactly the isolated-false line** that
every filter in this programme deliberately nulls (and must null: the grass).
A $P_1$ true stub is spectrally *identical* to an isolated false segment, so it is
unrecoverable by **any** spectral filter — the fragment version of the same-length
degeneracy.

In [4]:
solC,_=qp.solve_classical(HAM)
resQ=qp.solve_qsvt(HAM,degree=40,spectral_bounds=(0.5,7.5))
sQ=qp.rescale_to_signal(np.asarray(resQ["sol"],float),solC,TAU)
pP4=design_line_comb_inverse(degree=40,s=S)
EXT=tuple(sorted(P4L.tolist()+P3L[[0,2]].tolist()+P2L.tolist()))   # + s±sqrt2, s±1 (NOT s)
pEXT=design_line_comb_inverse(degree=60,s=S,lines=EXT)

fig,ax=plt.subplots(1,2,figsize=(15.5,5))
lam=np.linspace(0.4,7.6,2500)
ax[0].plot(lam,pP4(lam),color="#6a3d9a",lw=2.2,label="P4 comb (production, deg 40)")
ax[0].plot(lam,pEXT(lam),color="#2166ac",lw=1.6,ls="--",label="extended comb (+fragment lines, deg 60)")
for L in P4L: ax[0].axvline(L,color="#1b7837",lw=1.6,alpha=0.85)
for L in P3L[[0,2]]: ax[0].axvline(L,color="#e08214",lw=1.4,ls=":",alpha=0.9)
for L in P2L: ax[0].axvline(L,color="#d6604d",lw=1.4,ls=":",alpha=0.9)
ax[0].axvline(S,color="#b2182b",lw=2.0,ls="-.",alpha=0.9)
ax[0].text(S+0.04,0.42,"P1 / isolated-false\nline λ=4 (never pass!)",fontsize=8.5,color="#b2182b")
ax[0].text(P4L[0],0.47,"P4 lines",fontsize=9,color="#1b7837",ha="center")
ax[0].text(P3L[0],0.40,"P3",fontsize=9,color="#e08214",ha="center")
ax[0].text(P2L[0],0.44,"P2",fontsize=9,color="#d6604d",ha="center")
ax[0].set_xlabel("eigenvalue λ"); ax[0].set_ylabel("filter p(λ)")
ax[0].set_title("(a) the comb vs the fragment eigenvalues",fontweight="bold"); ax[0].legend(fontsize=9); ax[0].set_ylim(-0.08,0.52)
# per-class efficiency under the production comb
effs=[]
for cl in ("P4-true","P3-true","P2-true","P1-true","mixed chain","tangle"):
    m=CLS==cl
    if m.sum(): effs.append((cl,(sQ[TIDX[m]]>TAU).mean(),int(m.sum())))
labels=[f"{c}\n(n={n_})" for c,_,n_ in effs]
ax[1].bar(labels,[e*100 for _,e,_ in effs],color=["#1b7837","#e08214","#d6604d","#b2182b","#999","#555"],edgecolor="k")
for i,(_,e,_n) in enumerate(effs): ax[1].text(i,e*100+1.5,f"{e*100:.0f}%",ha="center",fontweight="bold")
ax[1].set_ylabel("segment efficiency (%)"); ax[1].set_ylim(0,108)
ax[1].set_title("(b) measured per-class efficiency, P4 comb (T=400, 1% drop)",fontweight="bold")
fig.tight_layout()
for ext,dpi in (("pdf",600),("png",300)): fig.savefig(OUT/f"drop_spectrum_and_class_eff.{ext}",dpi=dpi,bbox_inches="tight",facecolor="white")
plt.show(); print("saved drop_spectrum_and_class_eff")

saved drop_spectrum_and_class_eff


## 4. The fix — fragment lines + the hit-occupancy gate

**Step 1 (spectral):** add the $P_3$/$P_2$ fragment lines to the comb (never λ = 4).
This recovers the fragments — but their eigenvalues are shared by longer **false**
chains (e.g. $P_5$ has modes at $s\pm1$, $P_8$ near $s\pm\sqrt2$), and a polynomial
acts on eigenvalues only, so the false twins flood back in.

**Step 2 (classical, the occupancy principle):** a recovered *true fragment* comes
from a broken track — its hits are used by **no other active segment**. A
re-admitted *false chain* lives on hits that the active parent tracks also use.
Gate every newly recovered segment on **hit-sharing with the base-active set**.
This is the Denby–Peterson "one segment per hit" bifurcation principle, applied as
a free classical post-filter instead of a Hamiltonian term.

In [5]:
qvE=QSVT(Aq,np.asarray(HAM.b,float).ravel(),poly=pEXT,spectral_bounds=(0.5,7.5))
solE,_=qvE.solve_statevector(); sE=qp.rescale_to_signal(np.asarray(solE,float),solC,TAU)
base=sQ>TAU; ext=sE>TAU; rec=ext&~base
seg_hits=np.asarray(HAM._segment_to_hit_ids)
hit2segs=collections.defaultdict(list)
for i,(h1,h2) in enumerate(seg_hits):
    hit2segs[h1].append(i); hit2segs[h2].append(i)
def shares_hit_with_active(i,active):
    c=LAB[i]
    return any(active[j] and LAB[j]!=c for h in seg_hits[i] for j in hit2segs[h])
share=np.zeros(N,bool)
for i in np.where(rec)[0]: share[i]=shares_hit_with_active(i,base)
gate=rec&~share; final=base|gate
rt,rf=rec&TRUTH,rec&~TRUTH
print(f"recovery set (extended \\ base): {rec.sum()} segments — {rt.sum()} true, {rf.sum()} false")
print(f"hits shared with a base-active segment:  true {share[rt].mean()*100:.1f}%   false {share[rf].mean()*100:.1f}%")
res_rows=[]
def addrow(name,act):
    e=(act&TRUTH).sum()/TRUTH.sum(); f=(act&~TRUTH).sum()/max(act.sum(),1)
    res_rows.append(dict(pipeline=name,eff_pct=round(e*100,1),far_pct=round(f*100,2)))
solCact=np.abs(solC)>TAU
res1=qp.solve_quantum if False else None
addrow("classical 1/λ",solCact)
addrow("QSVT P4 comb (production)",base)
addrow("+ fragment lines (deg 60)",ext)
addrow("+ occupancy gate  ★",final)
R=pd.DataFrame(res_rows); print(); print(R.to_string(index=False))

fig,ax=plt.subplots(1,2,figsize=(14.5,5))
# (a) recovery composition & the gate
cats=["recovered true\n(fragments)","re-admitted false\n(chain twins)"]
shared=[share[rt].sum(),share[rf].sum()]; unshared=[(~share)[rt&rec].sum() if False else (rt&~share).sum(),(rf&~share).sum()]
ax[0].bar(cats,[rt.sum(),rf.sum()],color=["#1b7837","#d6604d"],alpha=0.35,edgecolor="k",label="recovered by fragment lines")
ax[0].bar(cats,[(rt&~share).sum(),(rf&~share).sum()],color=["#1b7837","#d6604d"],edgecolor="k",label="pass the occupancy gate")
ax[0].set_yscale("log"); ax[0].set_ylabel("# segments (log)")
for i,(tot,kept) in enumerate([(rt.sum(),(rt&~share).sum()),(rf.sum(),(rf&~share).sum())]):
    ax[0].text(i,tot*1.15,f"{tot}",ha="center",fontweight="bold")
    ax[0].text(i,max(kept,0.4)*1.3,f"{kept} kept",ha="center",fontsize=9,color="k")
ax[0].set_title("(a) the gate separates fragments from false twins",fontweight="bold"); ax[0].legend(fontsize=9)
# (b) pipeline comparison
x=np.arange(len(R)); w=0.38
ax[1].bar(x-w/2,R.eff_pct,w,color="#1b7837",edgecolor="k",label="efficiency (%)")
ax[1].bar(x+w/2,R.far_pct,w,color="#d6604d",edgecolor="k",label="false rate (%)")
for i,r in R.iterrows():
    ax[1].text(i-w/2,r.eff_pct+1,f"{r.eff_pct:.0f}",ha="center",fontsize=9,fontweight="bold")
    ax[1].text(i+w/2,r.far_pct+1,f"{r.far_pct:.2g}",ha="center",fontsize=9,fontweight="bold")
ax[1].set_xticks(x); ax[1].set_xticklabels(R.pipeline,fontsize=8.5)
ax[1].set_ylabel("%"); ax[1].set_ylim(0,112)
ax[1].set_title("(b) T=400, 1% drop: efficiency recovered at ~zero false cost",fontweight="bold"); ax[1].legend(fontsize=9)
fig.tight_layout()
for ext_,dpi in (("pdf",600),("png",300)): fig.savefig(OUT/f"drop_fix_occupancy_gate.{ext_}",dpi=dpi,bbox_inches="tight",facecolor="white")
plt.show(); print("saved drop_fix_occupancy_gate")

recovery set (extended \ base): 1250 segments — 49 true, 1201 false
hits shared with a base-active segment:  true 0.0%   false 99.4%

                 pipeline  eff_pct  far_pct
            classical 1/λ     97.9     2.37
QSVT P4 comb (production)     93.0     0.00
+ fragment lines (deg 60)     96.1    44.50
      + occupancy gate  ★     96.1     0.47


saved drop_fix_occupancy_gate


## 5. Verdict

- **Why the drop hurts:** 1 % hit inefficiency breaks ~5 % of tracks into $P_3$ /
  $P_2$ / $P_1$ fragments whose eigenvalues sit off the four $P_4$ comb lines —
  the comb erases them (per-class efficiency: $P_4$ 100 %, fragments 0 %). That is
  the entire efficiency loss.
- **What is fixable:** the $P_3$/$P_2$ fragments. Spectrally re-admitting their
  lines also re-admits degenerate false chains (a polynomial cannot separate
  identical eigenvalues), but the **hit-occupancy gate** can: 0 % of true fragments
  share hits with base-active segments vs ~99 % of the false twins. Net at T = 400,
  1 % drop: **efficiency 93.0 → 96.1 % at far 0.5 %** (classical: 98 % / 2.5 %).
- **What is not fixable:** the $P_1$ stubs sit exactly on the isolated-false line
  λ = s — spectrally identical to the grass; and tangle-embedded true segments
  (~2 % here) need the track-level treatment. These are the same degeneracy floors
  identified in the plan (Step A §4), now seen from the efficiency side.
- **The pattern worth keeping:** quantum spectral filter for the bulk +
  *classical occupancy post-gate* for the degenerate residue — the Denby–Peterson
  bifurcation principle applied where the spectrum is provably blind.